# GiftMind: AI Personality Analysis Training
This notebook contains the pipeline to clean the MBTI dataset and train a TF-IDF + Logistic Regression model for personality prediction.

In [ ]:
import pandas as pd
import numpy as np
import re
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

## 1. Data Loading
Upload your `mbti_1.csv` file or mount your Google Drive.

In [ ]:
# Option A: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_PATH = '/content/drive/MyDrive/mbti_1.csv'

# Option B: Local Upload (if dataset is in the same folder)
DATASET_PATH = 'mbti_1.csv'

## 2. Text Cleaning
Preprocessing the raw personality posts.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Loading dataset...")
df = pd.read_csv(DATASET_PATH)
print("Cleaning data...")
df['posts'] = df['posts'].apply(clean_text)

## 3. Training & Evaluation

In [ ]:
X = df['posts']
y = df['type']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=2000, multi_class='multinomial'))
])

print("Training model...")
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

## 4. Saving the Model

In [ ]:
joblib.dump(pipeline, 'mbti_model_improved.joblib')
print("Model saved as mbti_model_improved.joblib")